In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/fluentiq_scored_dataset_v1.csv")

print(df.shape)
df.head()

(100, 28)


,audio_file,transcript,duration_seconds,word_count,words_per_minute,average_rms_energy,average_zero_crossing_rate,spectral_centroid,spectral_bandwidth,mfcc_1,...,mfcc_10,mfcc_11,mfcc_12,mfcc_13,fluency_score,confidence_score,clarity_score,vocabulary_score,overall_score,communication_level
0,..\data\raw\librispeech\LibriSpeech\test-clean...,HE HOPED THERE WOULD BE STEW FOR DINNER TURNIP...,10.44,28,161.00,0.034645,0.127497,1704.13,1476.58,-300.5381,...,-0.9489,-0.2123,1.6172,-5.0724,75,69.29,87.25,84,77.82,Intermediate
1,..\data\raw\librispeech\LibriSpeech\test-clean...,STUFF IT INTO YOU HIS BELLY COUNSELLED HIM,3.27,8,146.56,0.019861,0.107820,1741.59,1628.54,-364.2475,...,-2.4162,4.9232,1.5388,1.1172,90,40.00,89.22,50,69.34,Intermediate
2,..\data\raw\librispeech\LibriSpeech\test-clean...,AFTER EARLY NIGHTFALL THE YELLOW LAMPS WOULD L...,6.62,18,163.02,0.038250,0.087339,1390.03,1353.31,-303.5464,...,-1.2306,0.2887,-0.0212,-3.2952,75,76.50,91.27,54,74.43,Intermediate
3,..\data\raw\librispeech\LibriSpeech\test-clean...,HELLO BERTIE ANY GOOD IN YOUR MIND,2.68,7,156.72,0.063083,0.065348,1312.21,1378.24,-313.2429,...,-2.0147,-4.1345,-0.2080,-6.3018,90,100.00,93.47,50,85.19,Advanced
4,..\data\raw\librispeech\LibriSpeech\test-clean...,NUMBER TEN FRESH NELLY IS WAITING ON YOU GOOD ...,5.22,11,126.56,0.048197,0.096332,1548.08,1483.26,-298.8683,...,0.8973,-3.4142,1.5993,-3.0086,90,96.39,90.37,50,83.67,Advanced


In [2]:
feature_columns = [
    "duration_seconds",
    "word_count",
    "words_per_minute",
    "average_rms_energy",
    "average_zero_crossing_rate",
    "spectral_centroid",
    "spectral_bandwidth",
    "mfcc_1", "mfcc_2", "mfcc_3", "mfcc_4", "mfcc_5",
    "mfcc_6", "mfcc_7", "mfcc_8", "mfcc_9", "mfcc_10",
    "mfcc_11", "mfcc_12", "mfcc_13"
]

X = df[feature_columns]

y_class = df["communication_level"]
y_reg = df["overall_score"]

print("X shape:", X.shape)
print("Classification target:", y_class.shape)
print("Regression target:", y_reg.shape)

X shape: (100, 20)
Classification target: (100,)
Regression target: (100,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_class, y_test_class = train_test_split(
    X,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

_, _, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 80
Testing samples: 20


In [4]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

classifier.fit(X_train, y_train_class)

print("Classification model trained successfully.")

Classification model trained successfully.


In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_class = classifier.predict(X_test)

accuracy = accuracy_score(y_test_class, y_pred_class)

print("Accuracy:", round(accuracy * 100, 2), "%")
print()
print(classification_report(y_test_class, y_pred_class))

Accuracy: 80.0 %

              precision    recall  f1-score   support

    Advanced       0.78      0.88      0.82         8
    Beginner       0.00      0.00      0.00         1
Intermediate       0.82      0.82      0.82        11

    accuracy                           0.80        20
   macro avg       0.53      0.56      0.55        20
weighted avg       0.76      0.80      0.78        20



c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Sanjiv\Desktop\SRM\PROJECTS\FluentIQ\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

## Train Regression Model

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

regressor.fit(X_train, y_train_reg)

y_pred_reg = regressor.predict(X_test)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 2))

MAE: 6.89
MSE: 72.1
RMSE: 8.49
R2 Score: 0.07


In [7]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(classifier, MODEL_DIR / "speech_level_classifier.pkl")
joblib.dump(regressor, MODEL_DIR / "wpm_regressor.pkl")

print("Models saved successfully.")

Models saved successfully.


In [8]:
loaded_classifier = joblib.load("../models/speech_level_classifier.pkl")
loaded_regressor = joblib.load("../models/wpm_regressor.pkl")

print("Models loaded successfully.")

Models loaded successfully.


In [9]:
sample = X_test.iloc[[0]]

predicted_level = loaded_classifier.predict(sample)[0]
predicted_wpm = loaded_regressor.predict(sample)[0]

print("Predicted Level :", predicted_level)
print("Predicted WPM   :", round(predicted_wpm, 2))

Predicted Level : Intermediate
Predicted WPM   : 76.9
